In [0]:
%sql
CREATE CATALOG IF NOT EXISTS V_Commerce;

In [0]:
%sql
USE SCHEMA DEFAULT;

CREATE VOLUME IF NOT EXISTS landing;

In [0]:
%sql
USE CATALOG V_Commerce;

CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:
from pyspark.sql import functions as F
import re
import unicodedata 

In [0]:
catalog = "V_Commerce"
bronze_schema_name = "bronze"

In [0]:
def get_invalid_columns(df):
    invalid_pattern = re.compile(r'[ ,;{}()\n\t=.]')
    invalid_columns = []
    
    for col in df.columns:
        normalized = unicodedata.normalize('NFKD', col).encode('ascii', 'ignore').decode('utf-8')

        if invalid_pattern.search(col) or col != normalized or any(char.isupper() for char in col):
            invalid_columns.append(col)
            
    return invalid_columns

In [0]:
def clean_column_names(df):
    new_columns = []
    
    for col in df.columns:
        name = unicodedata.normalize('NFKD', col).encode('ascii', 'ignore').decode('utf-8')
        name = re.sub(r'[ ,;{}()\n\t=.]+', '_', name)
        name = re.sub(r"_+", "_", name).strip('_').lower()
        new_columns.append(name)
        
    return df.toDF(*new_columns)

In [0]:
def ingest_csv(nome_arquivo, nome_tabela):
    try:
        landing_path = f"/Volumes/v_commerce/default/landing/{nome_arquivo}"

        df = spark.read.csv(landing_path, header=True, inferSchema=True)

        if df.count() == 0:
            print(f"O arquivo {nome_arquivo} está vazio ou não foi lido corretamente.")
            return 

        invalid_cols = get_invalid_columns(df)
        
        if invalid_cols:
            print(f"⚠️ Atenção: Encontradas {len(invalid_cols)} coluna(s) fora do padrão em {nome_arquivo}.")
            print("⏳ Corrigindo colunas automaticamente em memória...")
            df = clean_column_names(df)
        else:
            print(f"✅ As colunas do arquivo {nome_arquivo} já estão no padrão correto.")

        df_with_metadata = df.withColumn("timestamp_ingestion", F.current_timestamp())

        full_table_name = f"`{catalog}`.{bronze_schema_name}.{nome_tabela}"

        df_with_metadata.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(full_table_name)

        print(f"✅ Tabela {full_table_name} criada com sucesso\n")

    except Exception as E:
        print(f"❌ Falha ao processar o arquivo {nome_arquivo}: {str(E)}")

In [0]:
ingest_csv("pedidos.csv", "tb_pedidos") 
ingest_csv("avaliacoes.csv", "tb_avaliacoes")
ingest_csv("clientes.csv", "tb_clientes")
ingest_csv("catalogo_produtos.csv", "tb_catalogo_produtos")
ingest_csv("clickstream.csv", "tb_clickstream")
ingest_csv("suporte_tickets.csv", "tb_suporte_tickets")

✅ As colunas do arquivo pedidos.csv já estão no padrão correto.
✅ Tabela `V_Commerce`.bronze.tb_pedidos criada com sucesso

✅ As colunas do arquivo avaliacoes.csv já estão no padrão correto.
✅ Tabela `V_Commerce`.bronze.tb_avaliacoes criada com sucesso

✅ As colunas do arquivo clientes.csv já estão no padrão correto.
✅ Tabela `V_Commerce`.bronze.tb_clientes criada com sucesso

✅ As colunas do arquivo catalogo_produtos.csv já estão no padrão correto.
✅ Tabela `V_Commerce`.bronze.tb_catalogo_produtos criada com sucesso

✅ As colunas do arquivo clickstream.csv já estão no padrão correto.
✅ Tabela `V_Commerce`.bronze.tb_clickstream criada com sucesso

✅ As colunas do arquivo suporte_tickets.csv já estão no padrão correto.
✅ Tabela `V_Commerce`.bronze.tb_suporte_tickets criada com sucesso



In [0]:
df_pedidos = spark.table("V_commerce.bronze.tb_pedidos")
#df_pedidos.display()

In [0]:
df_avaliacoes = spark.table("V_commerce.bronze.tb_avaliacoes")
#df_avaliacoes.display()